
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [6]:
import os, json

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001AEB6C1F610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AEB6CE8050>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### Lets build a recipe based system

In [4]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    name:str=Field(description="The name of the dish")
    cuisine:str=Field(description="The cuisine style of the dish")
    prep_time:int=Field(description="Preparation time in minutes")
    ingredients:list[str]=Field(description="List of ingredients needed for the dish")

In [3]:
# not an model invocation, with_structured_output(Recipe) just a runnable
model_with_structure=model.with_structured_output(Recipe)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001AEB69BA660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AEB69BB0E0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Recipe', 'description': '', 'parameters': {'properties': {'name'

In [5]:
model.invoke("Tell me about khachapuri, the Georgian cheese-filled bread")

AIMessage(content='**Khachapuri – the iconic Georgian cheese‑filled bread**\n\n| Feature | What it means |\n|---------|---------------|\n| **Name** | *Khachapuri* (ხაჭაპური) – literally “cheese‑bread” in Georgian. |\n| **Origin** | Ancient Georgia (the Caucasus). It is a staple of Georgian hospitality and has been baked in the region for centuries, long before modern borders existed. |\n| **Cultural status** | Considered a national dish, often served at celebrations, family gatherings, and in taverns (known locally as *supra*). It’s a symbol of warmth, generosity, and culinary pride. |\n\n---\n\n## 1. The dough and the cheese\n\n| Ingredient | Typical Georgian version |\n|------------|------------------------|\n| **Dough** | Made from wheat flour, water, yeast, a pinch of salt, and sometimes a little sugar. It is kneaded until smooth and left to rise for 1–2\u202fh. |\n| **Cheese** | The classic cheese is **sulguni** (a hard, salty cheese made from cow or goat milk). It’s often mixed w

In [7]:
response=model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")

print(json.dumps(response.model_dump(),indent=2, default=str))

{
  "name": "Khachapuri",
  "cuisine": "Georgian",
  "prep_time": 45,
  "ingredients": [
    "500g all-purpose flour",
    "1 tsp salt",
    "1 tsp sugar",
    "1 tsp instant yeast",
    "300ml warm water",
    "2 tbsp olive oil",
    "200g cheese blend (suluguni, feta, mozzarella)",
    "1 egg yolk",
    "1 tbsp butter",
    "1 egg (for egg wash)"
  ]
}


### MEssage output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    """A recipe with details."""
    name: str = Field(..., description="The name of the dish")
    cuisine: str = Field(..., description="The cuisine style of the dish")
    prep_time: int = Field(..., description="Preparation time in minutes")
    ingredients: list[str] = Field(..., description="List of ingredients needed for the dish")

model_with_structure = model.with_structured_output(Recipe, include_raw=True)  

response = model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")


In [10]:
response
# print(json.dumps(
#     {
#         "raw": response["raw"].model_dump(),
#         "parsed": response["parsed"].model_dump(),
#         "parsing_error": response["parsing_error"],
#     },
#     indent=2,
#     default=str,
#     ensure_ascii=False,
# ))

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants recipe details for khachapuri. They want the recipe details. We have a tool to provide the recipe. We should call the function. The function expects cuisine, ingredients, name, prep_time. So we need to supply those. The cuisine: Georgian. Name: Khachapuri. Ingredients: list of ingredients. Prep time: maybe 60 minutes? Let's approximate. Provide a typical recipe. Let's think typical ingredients: For dough: flour, yeast, water, sugar, salt, oil. For filling: cheese (like sulguni or feta), eggs, butter. Additional toppings: eggs, butter, etc. Provide typical. We should call the function.", 'tool_calls': [{'id': 'fc_24fa9494-f7b2-43e1-90b1-9c9ee95e7147', 'function': {'arguments': '{"cuisine":"Georgian","ingredients":["2 cups all-purpose flour","1 packet active dry yeast (7g)","1 tsp sugar","1 tsp salt","1 tbsp olive oil","3/4 cup warm water","1 cup shredded sulguni or a mix of sulguni and feta cheese","1 

### Nested Structure

In [11]:
from pydantic import BaseModel, Field


class Ingredient(BaseModel):
    item: str
    quantity: str

class RecipeDetails(BaseModel):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for khachapuri, the Georgian cheese-filled bread")
response

RecipeDetails(name='Khachapuri', cuisine='Georgian', ingredients=[Ingredient(item='all-purpose flour', quantity='500g (about 4 cups)'), Ingredient(item='active dry yeast', quantity='7g (1 packet)'), Ingredient(item='warm milk', quantity='250ml (1 cup)'), Ingredient(item='sugar', quantity='1 tsp'), Ingredient(item='salt', quantity='1 tsp'), Ingredient(item='unsalted butter, melted', quantity='30g (2 tbsp)'), Ingredient(item='egg, beaten', quantity='1 (for brushing dough)'), Ingredient(item='sulghuni cheese (or a blend of mozzarella and feta)', quantity='250g (about 9 oz)'), Ingredient(item='extra butter, melted', quantity='30g (2 tbsp)')], steps=['1. In a small bowl, dissolve the yeast in the warm milk with a pinch of sugar. Let sit for 5–10 minutes until frothy.', '2. In a large mixing bowl, combine the flour, remaining sugar, and salt. Make a well in the center and add the yeast mixture, melted butter, and 1 beaten egg. Mix until a shaggy dough forms.', '3. Turn the dough onto a light

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [9]:
from typing import Annotated

from typing_extensions import TypedDict


class RecipeDict(TypedDict):
    """A recipe with details."""
    name: Annotated[str, ..., "The name of the dish"]
    cuisine: Annotated[str, ..., "The cuisine style of the dish"]
    prep_time: Annotated[int, ..., "Preparation time in minutes"]
    ingredients: Annotated[list[str], ..., "List of ingredients needed for the dish"]


model_withtypedict=model.with_structured_output(RecipeDict)
response=model_withtypedict.invoke("Please provide the recipe details for pizza")
response

{'cuisine': 'Italian',
 'ingredients': ['Pizza dough',
  'Tomato sauce',
  'Mozzarella cheese',
  'Fresh basil leaves',
  'Olive oil',
  'Salt',
  'Pepper'],
 'name': 'Classic Margherita Pizza',
 'prep_time': 30}

In [15]:
bad = RecipeDict(
    name=123,                 # should be str
    cuisine=None,             # should be str
    prep_time="not a number", # should be int
    ingredients="cheese",     # should be list[str]
)
print(bad)   # no error

{'name': 123, 'cuisine': None, 'prep_time': 'not a number', 'ingredients': 'cheese'}


In [13]:
class Ingredient(TypedDict):  # noqa: F811
    item: str
    quantity: str

class RecipeDetails(TypedDict):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for pancakes")
response

{'cuisine': 'American',
 'ingredients': [{'item': 'All-purpose flour', 'quantity': '1 1/2 cups'},
  {'item': 'Baking powder', 'quantity': '3 1/2 teaspoons'},
  {'item': 'Salt', 'quantity': '1 teaspoon'},
  {'item': 'White sugar', 'quantity': '1 tablespoon'},
  {'item': 'Milk', 'quantity': '1 1/4 cups'},
  {'item': 'Egg', 'quantity': '1 large'},
  {'item': 'Melted butter', 'quantity': '3 tablespoons'},
  {'item': 'Vanilla extract', 'quantity': '1 teaspoon'}],
 'name': 'Classic Pancakes',
 'prep_time': 15,
 'steps': ['In a large mixing bowl, sift together the flour, baking powder, salt, and sugar.',
  'Make a well in the center and pour in the milk, egg, melted butter, and vanilla. Mix until just combined; a few lumps are fine.',
  'Heat a lightly oiled griddle or frying pan over medium-high heat.',
  'Pour or scoop the batter onto the griddle, using about 1/4 cup for each pancake. Cook until bubbles form on the surface and the edges look set, about 2-3 minutes.',
  'Flip with a spatula 

In [14]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2025-08-05',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [16]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")

In [17]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029', additional_kwargs={}, response_metadata={}, id='63d7fdf3-dcb0-4c8b-b018-158c92520add'),
  AIMessage(content='{"name":"Elena Marquez","email":"elena.marquez@brightleaf-consulting.com","phone":"(312) 774-5029"}', additional_kwargs={'reasoning_content': 'We need to output JSON in the format ContactInfo. Must be compact JSON, all fields required: name, email, phone. Provide values. Ensure JSON is valid. Let\'s produce: {"name":"Elena Marquez","email":"elena.marquez@brightleaf-consulting.com","phone":"(312) 774-5029"}'}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 247, 'total_tokens': 364, 'completion_time': 0.12010421, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.01209366, 'prompt_tokens_details': None, 'queue_time': 0.366472598, 'total_time': 0.13219787}, 'model_name': 'openai/gpt-oss-20b', 'sys

In [18]:
result["structured_response"]

ContactInfo(name='Elena Marquez', email='elena.marquez@brightleaf-consulting.com', phone='(312) 774-5029')

In [19]:
## Dataclass

from dataclasses import dataclass

from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result["structured_response"]

ContactInfo(name='Elena Marquez', email='elena.marquez@brightleaf-consulting.com', phone='(312) 774-5029')

### dataclass and pydantic in same example

In [20]:
from dataclasses import dataclass

from pydantic import BaseModel, ValidationError


@dataclass
class ContactDC:
    name: str
    email: str
    phone: str

class ContactPD(BaseModel):
    name: str
    email: str
    phone: str

print(ContactDC(name=123, email=None, phone=["x"]))   # works, no error

try:
    ContactPD(name=123, email=None, phone=["x"])
except ValidationError as e:
    print(e)                                           # raises

ContactDC(name=123, email=None, phone=['x'])
3 validation errors for ContactPD
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
email
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
phone
  Input should be a valid string [type=string_type, input_value=['x'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
